<a href="https://colab.research.google.com/github/FONDECYTACC/cons2025/blob/main/best_subset_based_on_disc_jan26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Best predictors based on discrimination, permutation

## 0. Package loading and installation

In [ ]:
#@title 🛠️ Environment Setup & Helper Functions { display-mode: "form" }

# 1. Reset environment and Clear Memory
%reset -f
import gc
import re
import numpy as np
import pandas as pd
from google.colab import data_table

# 2. Install necessary libraries (silent mode)
!pip install -q scikit-survival miceforest xlsxwriter

# 3. Imports
from sksurv.metrics import concordance_index_ipcw, brier_score, integrated_brier_score
from sksurv.util import Surv

# 4. CUSTOM HELPER FUNCTIONS (R-style)

def glimpse(df, max_width=80):
    """View dataframe structure similar to R's glimpse()"""
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    for col in df.columns:
        dtype = df[col].dtype
        preview = df[col].astype(str).head(5).tolist()
        preview_str = ", ".join(preview)
        if len(preview_str) > max_width:
            preview_str = preview_str[:max_width] + "..."
        print(f"{col:<30} {str(dtype):<15} {preview_str}")

def tabyl(series):
    """Frequency table similar to R's janitor::tabyl()"""
    counts = series.value_counts(dropna=False)
    props = series.value_counts(normalize=True, dropna=False)
    return pd.DataFrame({
        "value": counts.index,
        "n": counts.values,
        "percent": props.values
    }).sort_values("value")

def clean_names(df):
    """Clean column names similar to R's janitor::clean_names()"""
    new_cols = []
    for col in df.columns:
        col = col.lower()
        col = re.sub(r"[^\w]+", "_", col)
        col = col.strip("_")
        new_cols.append(col)
    df.columns = new_cols
    return df

# 5. Enable Interactive Tables for better head() visualization
data_table.enable_dataframe_formatter()

gc.collect()
print("✅ Environment reset. Libraries installed. Helper functions loaded.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 14.8 MB/s eta 0:00:00
✅ Environment reset. Libraries installed. Helper functions loaded.


In [ ]:
from google.colab import userdata

# Names of the objects (and secrets)
object_names = [
    "imputations_list_jan26",
    "imputation_nodum_1",
    "X_reduced_imp0",
    "imputation_1"
]
for name in object_names:
    file_id = userdata.get(name)   # secret stored with this key
    if file_id is None:
        raise ValueError(f"No file_id found in userdata for key '{name}'")

    output_file = f"{name}.pkl"
    print(f"Downloading {name} -> {output_file}")
    !gdown {file_id} --output {output_file} --quiet

Load in python

In [ ]:
import pandas as pd

# Example: load only imputation 1
imputation_nodum_1 = pd.read_parquet("/content/imputation_nodum_1.pkl")
X_reduced_imp0 = pd.read_parquet("/content/X_reduced_imp0.pkl")
imputation_1 = pd.read_parquet("/content/imputation_1.pkl")

# Quick check
glimpse(imputation_nodum_1)
glimpse(imputation_1)
glimpse(X_reduced_imp0)



Rows: 88504 | Columns: 41
readmit_time_from_adm_m        float64         84.93548387096774, 12.833333333333334, 13.733333333333333, 11.966666666666667, 1...
death_time_from_adm_m          float64         84.93548387096774, 87.16129032258064, 117.2258064516129, 98.93548387096774, 37.9...
adm_age_rec3                   float64         31.53, 20.61, 42.52, 60.61, 45.08
porc_pobr                      float64         0.175679117441177, 0.187835901975632, 0.130412444472313, 0.133759185671806, 0.08...
dit_m                          float64         15.967741935483872, 5.833333333333334, 0.4752688172043005, 6.966666666666667, 6....
sex_rec                        category        man, man, man, woman, man
tenure_status_household        category        stays temporarily with a relative, owner/transferred dwellings/pays dividends, s...
cohabitation                   category        alone, family of origin, with couple/children, with couple/children, family of o...
sub_dep_icd10_status           cat

In [ ]:
import pickle
import pandas as pd
import numpy as np

file_path = '/content/imputations_list_jan26.pkl'

with open(file_path, 'rb') as f:
    imputations_list_jan26 = pickle.load(f)

print(f"Successfully loaded '{file_path}'\n")
print(f"Type of loaded object: {type(imputations_list_jan26)}")

if isinstance(imputations_list_jan26, list) and len(imputations_list_jan26) > 0:
    print("First element type:", type(imputations_list_jan26[0]))
    if isinstance(imputations_list_jan26[0], dict):
        print("First element keys:", imputations_list_jan26[0].keys())
    elif isinstance(imputations_list_jan26[0], (pd.DataFrame, np.ndarray)):
        print("First element shape:", imputations_list_jan26[0].shape)


Successfully loaded '/content/imputations_list_jan26.pkl'

Type of loaded object: <class 'list'>
First element type: <class 'pandas.core.frame.DataFrame'>
First element shape: (88504, 56)


This code block:

1.  **Imports the `pickle` library**: This library implements binary protocols for serializing and de-serializing a Python object structure.
2.  **Specifies the `file_path`**: It points to the `.pkl` file you selected.
3.  **Opens the file in binary read mode (`'rb'`)**: This is necessary for loading pickle files.
4.  **Loads the object**: `pickle.load(f)` reads the serialized object from the file and reconstructs it in memory.
5.  **Prints confirmation and basic information**: It verifies that the file was loaded and shows the type of the loaded object, and some details about the first element if it's a list containing common data structures.

#### Compare databases (transformed and original)

Inspect and compare the column names of two datasets: the first imputation from imputations_list_jan26 (which likely contains dummy variables) and imputation_nodum_1 (which, as its name suggests, probably doesn't have dummy variables).


In [ ]:
# Inspect columns of the first imputation
cols_first_imp = imputations_list_jan26[0].columns.tolist()
print("First imputation columns:", cols_first_imp[:10], "... total:", len(cols_first_imp))

# Inspect columns of imputation_no_dum
cols_nodum = imputation_nodum_1.columns.tolist()
print("No-dum columns:", cols_nodum[:10], "... total:", len(cols_nodum))

# Compare overlap
common_cols = set(cols_first_imp).intersection(cols_nodum)
missing_in_imp = [c for c in cols_nodum if c not in cols_first_imp]
missing_in_nodum = [c for c in cols_first_imp if c not in cols_nodum]

print("Common columns:", len(common_cols))
print("Missing in imputations_list_jan26:", missing_in_imp)

First imputation columns: ['adm_age_rec3', 'porc_pobr', 'dit_m', 'tenure_status_household', 'prim_sub_freq_rec', 'national_foreign', 'urbanicity_cat', 'ed_attainment_corr', 'evaluacindelprocesoteraputico', 'eva_consumo'] ... total: 56
No-dum columns: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'adm_age_rec3', 'porc_pobr', 'dit_m', 'sex_rec', 'tenure_status_household', 'cohabitation', 'sub_dep_icd10_status', 'any_violence'] ... total: 41
Common columns: 24
Missing in imputations_list_jan26: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'sex_rec', 'cohabitation', 'sub_dep_icd10_status', 'any_violence', 'tr_outcome', 'adm_motive', 'first_sub_used', 'primary_sub_mod', 'tipo_de_vivienda_rec2', 'plan_type_corr', 'occupation_condition_corr24', 'marital_status_rec', 'readmit_event', 'death_event', 'center_id']


In [ ]:
# Inspect columns of the first imputation
cols_first_imp_raw = imputation_1.columns.tolist()
print("First imputation columns:", cols_first_imp_raw[:10], "... total:", len(cols_first_imp_raw))

# Compare overlap
common_cols_raw = set(cols_first_imp_raw).intersection(cols_nodum)
missing_in_imp_raw = [c for c in cols_nodum if c not in cols_first_imp_raw]

print("Common columns:", len(common_cols_raw))
print("Missing in imputations_list_jan26:", missing_in_imp_raw)
print(common_cols_raw)

First imputation columns: ['readmit_time_from_adm_m', 'death_time_from_adm_m', 'adm_age_rec3', 'porc_pobr', 'dit_m', 'national_foreign', 'ethnicity', 'dg_psiq_cie_10_instudy', 'dg_psiq_cie_10_dg', 'dx_f3_mood'] ... total: 78
Common columns: 16
Missing in imputations_list_jan26: ['sex_rec', 'tenure_status_household', 'cohabitation', 'sub_dep_icd10_status', 'any_violence', 'prim_sub_freq_rec', 'tr_outcome', 'adm_motive', 'first_sub_used', 'primary_sub_mod', 'tipo_de_vivienda_rec2', 'plan_type_corr', 'occupation_condition_corr24', 'marital_status_rec', 'urbanicity_cat', 'ed_attainment_corr', 'evaluacindelprocesoteraputico', 'eva_consumo', 'eva_fam', 'eva_relinterp', 'eva_ocupacion', 'eva_sm', 'eva_fisica', 'eva_transgnorma', 'center_id']
{'readmit_event', 'polysubstance_strict', 'adm_age_rec3', 'death_time_from_adm_m', 'dx_f_any_severe_mental', 'dit_m', 'ethnicity', 'dg_psiq_cie_10_instudy', 'any_phys_dx', 'dg_psiq_cie_10_dg', 'dx_f6_personality', 'readmit_time_from_adm_m', 'national_fore

In [ ]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars = ["adm_age_rec3", "porc_pobr", "dit_m"]

# Take one imputation (first element of the list) and merge with the no-dum dataset
df_imp = imputations_list_jan26[0]
df_nodum = imputation_nodum_1

merged_check = pd.merge(
    df_imp[key_vars],
    df_nodum[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check.shape[0]}")
print("Preview of merged check:")
print(merged_check.head())

#drop merge
del merged_check

Merged rows: 88516
Preview of merged check:
   adm_age_rec3  porc_pobr      dit_m
0         31.53   0.175679  15.967742
1         20.61   0.187836   5.833333
2         42.52   0.130412   0.475269
3         60.61   0.133759   6.966667
4         45.08   0.083189   6.903226


In [ ]:
import pandas as pd

# Example: choose a combination of variables that uniquely identify rows
key_vars_raw = ['dit_m',
            'readmit_time_from_adm_m',
            'death_time_from_adm_m',
            'adm_age_rec3']
# Take one imputation (first element of the list) and merge with the no-dum dataset
df_raw = imputation_1

merged_check_raw = pd.merge(
    df_imp[key_vars],
    df_raw[key_vars],
    on=key_vars,
    how="inner"
)

print(f"Merged rows: {merged_check_raw.shape[0]}")
print("Preview of merged check:")
print(merged_check_raw.head())
print(f"{(merged_check_raw.shape[0] / imputation_1.shape[0] * 100):.2f}%")
#drop merge
del merged_check_raw

Merged rows: 88516
Preview of merged check:
   adm_age_rec3  porc_pobr      dit_m
0         31.53   0.175679  15.967742
1         20.61   0.187836   5.833333
2         42.52   0.130412   0.475269
3         60.61   0.133759   6.966667
4         45.08   0.083189   6.903226
100.01%


### Create bins for followup (landmarks)

This code prepares your data for survival analysis. It extracts the time until an event (like readmission or death) and whether that event actually happened for each patient from the df_nodum dataset. Then, it automatically creates a set of important time points, called an 'evaluation grid', which are specific moments to assess the model's performance on both readmission and death outcomes.


In [ ]:
import numpy as np

# Required columns for survival outcomes
required = ["readmit_time_from_disch_m", "readmit_event",
            "death_time_from_disch_m", "death_event"]

# Check that df_raw has all required columns
missing = [c for c in required if c not in df_raw.columns]
if missing:
    raise KeyError(f"df_nodum is missing columns: {missing}")

# Create time/event arrays directly from df_raw
time_readm = df_raw["readmit_time_from_adm_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_adm_m"].to_numpy()
event_death = (df_nodum["death_event"].to_numpy() == 1)

print("Arrays created for df_raw:")
print("Readmission times:", time_readm[:5])
print("Readmission events:", event_readm[:5])
print("Death times:", time_death[:5])
print("Death events:", event_death[:5])

# Build evaluation grids (quantiles of event times)
event_times_readm = time_readm[event_readm]
event_times_death = time_death[event_death]

if len(event_times_readm) < 5 or len(event_times_death) < 5:
    raise ValueError("Too few events in df_raw to build reliable time grids.")

times_eval_readm = np.unique(np.quantile(event_times_readm, np.linspace(0.05, 0.95, 50)))
times_eval_death = np.unique(np.quantile(event_times_death, np.linspace(0.05, 0.95, 50)))

print("Eval times (readmission):", times_eval_readm[:5], "...", times_eval_readm[-5:])
print("Eval times (death):", times_eval_death[:5], "...", times_eval_death[-5:])

Arrays created for df_raw:
Readmission times: [84.93548387 12.83333333 13.73333333 11.96666667 14.25806452]
Readmission events: [False  True  True  True  True]
Death times: [ 84.93548387  87.16129032 117.22580645  98.93548387  37.93548387]
Death events: [False False False False False]
Eval times (readmission): [3.93548387 4.77419355 5.45058701 6.06492649 6.67741935] ... [54.44173469 58.41566162 63.23333333 68.54767171 74.68983871]
Eval times (death): [4.16290323 5.43022383 6.68564845 8.24254115 9.77961817] ... [81.92700461 85.41186103 88.78518762 93.5538183  99.21935484]


# 3. “Best predictors” (variable importance) based on discrimination
* Inside each imputed dataset, we run k-fold CV, fit Coxnet on the training folds, and compute Uno’s C-index on the test folds.
* For each fold, we computed permutation importance by shuffling one predictor at a time in the test set, recomputing the C-index, and measuring the drop.
* We then pooled all these drops across folds and imputations, so `mean_drop_cindex` summarized how much that predictor hurts out-of-sample C-index on average, while respecting both multiple imputation and cross-validation.
* Sorting by `mean_drop_cindex` and taking the top 20 output the most influential predictors in a way that is robust to missing data and optimistic bias.


First, we eliminated inmortal time bias (dead patients look like without readmission).

This correction is essentially the Cause-Specific Hazard preparation. It is the correct way to handle Aim 3 unless you switch to a Fine-Gray model (which treats death as a specific type of event 2, rather than censoring 0). For RSF/Coxnet, censoring 0 is the correct approach.

In [ ]:
import numpy as np

# Step 1. Extract survival outcomes directly from df_raw
time_readm = df_raw["readmit_time_from_adm_m"].to_numpy()
event_readm = (df_raw["readmit_event"].to_numpy() == 1)

time_death = df_raw["death_time_from_adm_m"].to_numpy()
event_death = (df_raw["death_event"].to_numpy() == 1)

# Step 2. Build structured arrays (Surv objects)
y_surv_readm = np.empty(len(time_readm), dtype=[("event", "?"), ("time", "<f8")])
y_surv_readm["event"] = event_readm
y_surv_readm["time"] = time_readm

y_surv_death = np.empty(len(time_death), dtype=[("event", "?"), ("time", "<f8")])
y_surv_death["event"] = event_death
y_surv_death["time"] = time_death

# Step 3. Replicate across imputations
n_imputations = len(imputations_list_jan26)
y_surv_readm_list = [y_surv_readm for _ in range(n_imputations)]
y_surv_death_list = [y_surv_death for _ in range(n_imputations)]

import numpy as np

def correct_competing_risks(X_list, y_readm_list, y_death_list):
    """
    Adjust survival outcomes for competing risks (death vs. readmission).

    Parameters
    ----------
    X_list : list of pd.DataFrame
        Imputed predictor datasets (same rows across imputations).
    y_readm_list : list of structured arrays
        Surv(event, time) arrays for readmission.
    y_death_list : list of structured arrays
        Surv(event, time) arrays for death.

    Returns
    -------
    y_readm_corrected_list : list of structured arrays
        Corrected readmission outcomes (death treated as censoring).
    """
    corrected = []
    for y_readm, y_death in zip(y_readm_list, y_death_list):
        y_corr = y_readm.copy()
        # If patient died before readmission → censor at death time
        for i in range(len(y_corr)):
            if y_death["event"][i] and y_death["time"][i] < y_corr["time"][i]:
                y_corr["event"][i] = False
                y_corr["time"][i] = y_death["time"][i]
        corrected.append(y_corr)
    return corrected


# Step 4. Apply correction
y_surv_readm_list_corrected = correct_competing_risks(
    imputations_list_jan26,
    y_surv_readm_list,
    y_surv_death_list
)


In [ ]:
# Check type and length
type(y_surv_readm_list_corrected), len(y_surv_readm_list_corrected)

# Look at the first element
y_surv_readm_list_corrected[0][:5]   # first 5 rows
neg_times  = (y_surv_death_list[0]["time"] < 0).sum()
zero_times = (y_surv_death_list[0]["time"] == 0).sum()

print(f"Negative survival times: {neg_times}")
print(f"Zero survival times: {zero_times}")


Negative survival times: 0
Zero survival times: 1


### Optimized version

In [ ]:
#@title 📊 Feature Importance Analysis (Coxnet + MI+ CV) { display-mode: "form" }
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from joblib import Parallel, delayed

def permutation_importance_cindex_cv_mi(
    X_list,
    y_surv_list, # Changed to receive a list of y_surv objects
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.9,
    alpha_min_ratio=0.01,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,  # New: number of parallel jobs (-1 = all cores)
):
    """
    Multiple-imputation + k-fold CV permutation importance for Coxnet
    using Uno's C-index (out-of-sample).

    Parameters
    ----------
    X_list : list of pandas.DataFrame
        List of imputed design matrices (one per imputation).
        All must have the same rows and columns in the same order.
    y_surv_list : list of structured arrays
        List of Surv(event, time) structured arrays (one per imputation).
    alpha_idx : int
        Index along the Coxnet regularization path to use for predictions.
    n_splits : int
        Number of CV folds.
    n_repeats : int
        Number of permutations per feature per fold.
    random_state : int
        Seed for KFold and permutations.
    Other kwargs : CoxnetSurvivalAnalysis hyperparameters.
    n_jobs : int, optional
        Number of jobs for parallel processing of imputation-fold combinations.

    Returns
    -------
    baseline_cindex_mean : float
        Mean baseline CV Uno C-index across folds and imputations.
    baseline_cindex_sd   : float
        SD of baseline CV Uno C-index across folds and imputations.
    df_imp : pandas.DataFrame
        Feature-level permutation importance pooled over folds & imputations:
        columns = [feature, mean_drop_cindex, sd_drop_cindex, n_evals]
    """
    # Convert to NumPy arrays upfront for speed
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(float) for X in X_list]
    n_imputations = len(X_list)
    n = X_list[0].shape[0]

    # Precompute CV splits once (same indices used for all imputations)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(n)))

    # Function to compute baseline C-index and drops for one imputation-fold
    def compute_fold(d, fold_idx, train_idx, test_idx):
        print(f"  Imputation {d}, fold {fold_idx+1}/{n_splits}")
        X_imp = X_list[d]
        X_train = X_imp[train_idx, :]
        X_test = X_imp[test_idx, :]
        y_train = y_surv_list[d][train_idx] # Access the correct y_surv for the current imputation
        y_test = y_surv_list[d][test_idx]   # Access the correct y_surv for the current imputation

        # Local RNG with unique seed for this fold-imputation
        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=False,
            fit_baseline_model=True,
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        # Effective alpha index
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)

        # Baseline risks and C-index
        risk_baseline = model.predict(X_test, alpha=model.alphas_[eff_alpha_idx])
        res_base = concordance_index_ipcw(y_train, y_test, risk_baseline)
        cindex_baseline = float(res_base[0])

        # Permutation drops per feature (list of lists)
        fold_drops = [[] for _ in range(n_features)]
        for col_idx in range(n_features):
            for r in range(n_repeats):
                X_perm = X_test.copy()
                X_perm[:, col_idx] = local_rng.permutation(X_perm[:, col_idx])

                risk_perm = model.predict(X_perm, alpha=model.alphas_[eff_alpha_idx])
                res_perm = concordance_index_ipcw(y_train, y_test, risk_perm)
                cindex_perm = float(res_perm[0])
                fold_drops[col_idx].append(cindex_baseline - cindex_perm)

        return cindex_baseline, fold_drops

    print(f"\n=== {n_imputations} imputations – {n_splits}-fold CV permutation importance ===")

    # Parallelize over all imputation-fold combinations
    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_fold)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Collect results
    baseline_cindices = [res[0] for res in results]
    global_drops = [[] for _ in range(n_features)]
    for res in results:
        fold_drops = res[1]
        for col_idx in range(n_features):
            global_drops[col_idx].extend(fold_drops[col_idx])

    # Aggregate feature importances
    imp_rows = []
    for col_idx in range(n_features):
        arr = np.array(global_drops[col_idx])
        mean_drop = float(arr.mean()) if arr.size > 0 else np.nan
        sd_drop = float(arr.std(ddof=1)) if arr.size > 1 else 0.0
        imp_rows.append({
            "feature": feature_names[col_idx],
            "mean_drop_cindex": mean_drop,
            "sd_drop_cindex": sd_drop,
            "n_evals": int(arr.size),
        })

    df_imp_proc = pd.DataFrame(imp_rows)
    df_imp_proc = df_imp_proc.sort_values("mean_drop_cindex", ascending=False).reset_index(drop=True)

    # Baseline C-index summary
    baseline_arr = np.array(baseline_cindices)
    baseline_cindex_mean = float(baseline_arr.mean())
    baseline_cindex_sd = float(baseline_arr.std(ddof=1)) if baseline_arr.size > 1 else 0.0

    print("\n=== Baseline CV Uno C-index over imputations & folds ===")
    print(f"Mean ± SD: {baseline_cindex_mean:.4f} ± {baseline_cindex_sd:.4f}")

    return baseline_cindex_mean, baseline_cindex_sd, df_imp_proc

In [ ]:
#@title 🧬 Permutation Importance (Fixed Alpha) { display-mode: "form" }

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw
from joblib import Parallel, delayed

def permutation_importance_fixed_alpha(
    X_list,
    y_surv_list,
    alpha_idx=25,  # Using the index that gave you 0.608
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    n_jobs=-1
):
    # 1. Prepare Data (Convert to numpy arrays for speed/safety)
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_arrays = [x.values.astype(float) for x in X_list]

    # 2. Setup CV
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    def process_fold(d, fold_idx, train_idx, test_idx):
        X_train, X_test = X_arrays[d][train_idx], X_arrays[d][test_idx]
        y_train, y_test = y_surv_list[d][train_idx], y_surv_list[d][test_idx]

        # A. Fit Model
        model = CoxnetSurvivalAnalysis(
            l1_ratio=0.9,
            alpha_min_ratio=0.01,
            n_alphas=50,
            normalize=True,
            fit_baseline_model=True,
            max_iter=100000
        )
        model.fit(X_train, y_train)

        # B. Select Fixed Alpha
        # Ensure index doesn't exceed path length
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)
        fixed_alpha = model.alphas_[eff_alpha_idx]

        # C. Calculate Baseline Score
        risk_baseline = model.predict(X_test, alpha=fixed_alpha)
        c_baseline = concordance_index_ipcw(y_train, y_test, risk_baseline)[0]

        # D. Permutation Importance
        local_rng = np.random.RandomState(random_state + d*100 + fold_idx)
        drops = np.zeros((n_features, n_repeats))

        for col_idx in range(n_features):
            for r in range(n_repeats):
                X_perm = X_test.copy()
                # Shuffle one column
                X_perm[:, col_idx] = local_rng.permutation(X_perm[:, col_idx])

                # Predict & Score
                risk_perm = model.predict(X_perm, alpha=fixed_alpha)
                c_perm = concordance_index_ipcw(y_train, y_test, risk_perm)[0]

                # Drop = Baseline - Permuted
                drops[col_idx, r] = c_baseline - c_perm

        return c_baseline, drops

    print(f"Running Permutation Importance (Fixed Alpha {alpha_idx}) on {len(X_list)} imputations...")

    # 3. Run Parallel
    results = Parallel(n_jobs=n_jobs)(
        delayed(process_fold)(d, i, train, test)
        for d in range(len(X_list))
        for i, (train, test) in enumerate(kf.split(X_arrays[d]))
    )

    # 4. Aggregate
    baseline_cindices = [r[0] for r in results]
    all_drops = [r[1] for r in results]

    # Pool results
    pooled_drops = np.hstack(all_drops)
    mean_importances = np.mean(pooled_drops, axis=1)
    sd_importances = np.std(pooled_drops, axis=1)

    # Summary
    df_imp = pd.DataFrame({
        "feature": feature_names,
        "mean_drop_cindex": mean_importances,
        "sd_drop_cindex": sd_importances
    }).sort_values("mean_drop_cindex", ascending=False).reset_index(drop=True)

    print(f"\n=== Results ===")
    print(f"Baseline C-index: {np.mean(baseline_cindices):.4f} (Target: ~0.608)")

    return df_imp

# =======================================================
# EXECUTION
# =======================================================

# 1. Ensure y_surv_list contains Structured Arrays (not DataFrames)
# If you haven't fixed this yet, run this check:
if isinstance(y_surv_readm_list[0], pd.DataFrame):
    print("⚠️ Detected DataFrame outcomes. Converting to Structured Arrays...")
    y_surv_readm_list_fixed = []
    for df in y_surv_readm_list:
        y_surv_readm_list_fixed.append(
            np.array(list(zip(df["event"], df["time"])), dtype=[('event', '?'), ('time', '<f8')])
        )
    y_target = y_surv_readm_list_fixed
else:
    y_target = y_surv_readm_list


if isinstance(y_surv_death_list[0], pd.DataFrame):
    print("⚠️ Detected DataFrame outcomes. Converting to Structured Arrays...")
    y_surv_death_list_fixed = []
    for df in y_surv_death_list:
        y_surv_death_list_fixed.append(
            np.array(list(zip(df["event"], df["time"])), dtype=[('event', '?'), ('time', '<f8')])
        )
    y_target_death = y_surv_death_list_fixed
else:
    y_target_death = y_surv_death_list

### Execute

This calls the function with your data and parameters, unpacking the three returned values into variables:
- **`baseline_cidx_readm`**: Mean baseline C-index across all CV folds and imputations (a float, e.g., 0.75). Indicates overall model performance without permutations, with higher means better risk ranking.
- **`baseline_cidx_sd_readm`**: Standard deviation of the baseline C-index (measures variability/reliability across runs).
- **`df_imp_readm`**: A Pandas DataFrame ranking features by importance (sorted descending by mean C-index decrease after permutation). Columns likely include: "feature": Feature name, "mean_decrease_cidx": Average C-index drop (higher = more important, as permutation hurts performance more), "sd_decrease_cidx": Standard deviation of the drop, "n_evals": Number of permutations run for that feature (e.g., n_imputations * n_splits * n_repeats).

**Arguments Explained** (similar to IBS version, but no times_eval since C-index doesn't require a time grid):
- **`X_list`**= X_reduced_list: List of imputed feature matrices (Pandas DataFrames). Each is a version of your predictors with missing values filled differently.
- **`y_surv`**= y_surv_readm: Survival target (structured NumPy array with 'event' bool and 'time' float fields). E.g., time to readmission or censoring.
- **`alpha_idx`**=25: Index in Coxnet's regularization path (alphas from strong to weak). Picks a model complexity level (lower index = sparser model).
- **`n_splits`**=5: CV folds (5-fold = 80% train/20% test per fold).
- **`n_repeats`**=5: Permutations per feature per fold per imputation. Higher = more stable importances but longer runtime.

**Defaults** (not shown but from definition): random_state=2125 (for reproducibility), Coxnet params like l1_ratio=0.9 (mostly Lasso for sparsity), and n_jobs=-1 (parallelize on all cores).

In [ ]:
import time
from google.colab import data_table
data_table.enable_dataframe_formatter()

# Start timer
start_time = time.time()

baseline_cidx_readm, baseline_cidx_sd_readm, df_imp_readm = (
    permutation_importance_cindex_cv_mi(
        X_list=imputations_list_jan26,
        y_surv_list=y_surv_readm_list_corrected, # Changed to use the corrected list
        alpha_idx=49,# el último alpha (menos regularización) #25,
        n_splits=5,
        n_repeats=20,        # maybe 3 for speed; you can increase
        # More flexible regularization:
        l1_ratio=0.1,    # ✅ Más Ridge (mantiene variables) .5         # ← BALANCED Lasso/Ridge (was 0.9): 50% Lasso + 50% Ridge keeps more features active
        alpha_min_ratio=0.01,# ✅ 100x más pequeño - CRÍTICO  #.1     # ← LESS aggressive alpha range (was 0.01):  Focuses on less regularized models
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )
)

# End timer
end_time = time.time() # Print elapsed time in seconds
elapsed = end_time - start_time
print(f"Process completed in {elapsed/60:.2f} minutes")

# Top 20 predictors for readmission:
styled_table = df_imp_readm.head(20).style \
    .background_gradient(subset=['mean_drop_cindex'], cmap='Blues') \
    .format({'mean_drop_cindex': "{:.4f}", 'sd_drop_cindex': "{:.4f}"}) \
    .set_properties(**{'text-align': 'left', 'font-family': 'Arial'})

display(styled_table)

#~3 hrs. 21 min with 3 repeats


=== 5 imputations – 5-fold CV permutation importance ===

=== Baseline CV Uno C-index over imputations & folds ===
Mean ± SD: 0.5852 ± 0.0029
Process completed in 3170.49 seconds


,feature,mean_drop_cindex,sd_drop_cindex,n_evals
0,adm_age_rec3,0.0122,0.0035,500
1,sex_rec_woman,0.0090,0.0020,500
2,plan_type_corr_pg_pr,0.0047,0.0014,500
3,dit_m,0.0042,0.0030,500
4,plan_type_corr_m_pr,0.0029,0.0008,500
5,ethnicity,0.0028,0.0006,500
6,dg_psiq_cie_10_dg,0.0023,0.0008,500
7,primary_sub_mod_alcohol,0.0023,0.0013,500
8,sub_dep_icd10_status_drug_dependence,0.0018,0.0007,500
9,prim_sub_freq_rec,0.0014,0.0010,500


In [ ]:
# Start timer
start_time = time.time()

df_importance_readm = permutation_importance_fixed_alpha(
    X_list=imputations_list_jan26,  # Your Ordinal Encoded Data
    y_surv_list=y_target,         # Corrected Outcome List
    alpha_idx=49,                 # The index that worked in CV
    n_splits=5,
    n_repeats=20
)

# End timer
end_time = time.time() # Print elapsed time in seconds
elapsed = end_time - start_time
print(f"Process completed in {elapsed/60:.2f} minutes")

print("\nTop 15 Predictors for Readmission:")
styled_table = df_importance_readm.head(15).style \
    .background_gradient(subset=['mean_drop_cindex'], cmap='Blues') \
    .format({'mean_drop_cindex': "{:.4f}", 'sd_drop_cindex': "{:.4f}"}) \
    .set_properties(**{'text-align': 'left', 'font-family': 'Arial'})

display(styled_table)
#12 min, 5 repeat; 3 minutes with 1 repeat

Running Permutation Importance (Fixed Alpha 49) on 5 imputations...

=== Results ===
Baseline C-index: 0.6117 (Target: ~0.608)
Process completed in 3143.37 seconds

Top 15 Predictors for Readmission:


,feature,mean_drop_cindex,sd_drop_cindex
0,plan_type_corr_pg_pr,0.0115,0.0032
1,primary_sub_mod_alcohol,0.0099,0.0024
2,ethnicity,0.0098,0.0024
3,plan_type_corr_m_pr,0.0089,0.0022
4,sex_rec_woman,0.0080,0.0022
5,primary_sub_mod_marijuana,0.0067,0.0018
6,adm_age_rec3,0.0056,0.0017
7,ed_attainment_corr,0.0039,0.0016
8,tr_outcome_adm_discharge_rule_violation_undet,0.0036,0.0017
9,tr_outcome_referral,0.0031,0.0013


In [ ]:
import pandas as pd
import numpy as np
from sksurv.linear_model import CoxnetSurvivalAnalysis

# 1. Prepare Importance Data
# Rename the permutation importance column for clarity
df_top15 = pd.DataFrame(df_importance_readm.head(15))
df_top15 = df_top15.rename(columns={"mean_drop_cindex": "importance"})

# 2. Fit a Single Representative Model (Imputation 1)
# Using the best parameters identified during the cross-validation phase
print("Fitting representative model on Imputation 1 to get coefficients...")

# Ensure data sources are correctly aligned
X_rep = imputations_list_jan26[0]
y_rep = y_surv_readm_list[0]

model_rep = CoxnetSurvivalAnalysis(
    l1_ratio=0.1,
    alpha_min_ratio=0.01,
    n_alphas=50,
    normalize=True,
    fit_baseline_model=True,
    max_iter=100000,
    verbose=False
)
model_rep.fit(X_rep, y_rep)

# 3. Extract Coefficients at Alpha Index 25 (Safely)
# The error "Index out of bounds" happens if the solver finds fewer than 26 alphas.
# We check the actual length of the alpha path before indexing.
n_alphas_found = len(model_rep.alphas_)
target_alpha_idx = 25

if n_alphas_found > target_alpha_idx:
    # Use the requested index if available
    coeffs = model_rep.coef_[:, target_alpha_idx]
    print(f"Successfully extracted coefficients at alpha index {target_alpha_idx}.")
else:
    # Fallback to the last available alpha if index 25 doesn't exist
    coeffs = model_rep.coef_[:, -1]
    print(f"Warning: Only {n_alphas_found} alphas found. Using the last available index instead.")

feature_names = X_rep.columns.tolist()

# Create a reference dataframe for coefficient values
df_coefs = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coeffs
})

# 4. Merge Importance and Directional Data
# Combine the importance ranking with the model coefficients
df_final = df_top15.merge(df_coefs, on="feature", how="left")

# Define the clinical interpretation of the coefficient sign
# Positive (+) = Increases Hazard (Higher risk of early readmission)
# Negative (-) = Decreases Hazard (Protective factor/Lower risk)
df_final["Effect"] = df_final["coefficient"].apply(
    lambda x: "Increases Risk (Faster Readmission)" if x > 0 else "Protective (Slower Readmission)"
)

# 5. Display the Styled Results
print("\n=== Top 15 Predictors with Direction ===")
styled_table = df_final[["feature", "importance", "coefficient", "Effect"]].head(20).style \
    .background_gradient(subset=['importance'], cmap='Blues') \
    .format({'importance': "{:.4f}", 'coefficient': "{:.4f}"}) \
    .set_properties(**{'text-align': 'left', 'font-family': 'Arial'})

display(styled_table)

Fitting representative model on Imputation 1 to get coefficients...

=== Top 15 Predictors with Direction ===


,feature,importance,coefficient,Effect
0,plan_type_corr_pg_pr,0.0115,0.0000,Increases Risk (Faster Readmission)
1,primary_sub_mod_alcohol,0.0099,-0.0002,Protective (Slower Readmission)
2,ethnicity,0.0098,0.0000,Protective (Slower Readmission)
3,plan_type_corr_m_pr,0.0089,0.0011,Increases Risk (Faster Readmission)
4,sex_rec_woman,0.0080,0.0001,Increases Risk (Faster Readmission)
5,primary_sub_mod_marijuana,0.0067,0.0000,Protective (Slower Readmission)
6,adm_age_rec3,0.0056,-0.0000,Protective (Slower Readmission)
7,ed_attainment_corr,0.0039,0.0000,Protective (Slower Readmission)
8,tr_outcome_adm_discharge_rule_violation_undet,0.0036,0.0000,Protective (Slower Readmission)
9,tr_outcome_referral,0.0031,0.0000,Protective (Slower Readmission)


In [ ]:
# Start timer
start_time = time.time()

baseline_cidx_death, baseline_cidx_sd_death, df_imp_death = (
    permutation_importance_cindex_cv_mi(
        X_list=imputations_list_jan26,
        y_surv_list=y_surv_death_list, # Changed to use the list of death outcomes
        alpha_idx=49,
        n_splits=10,
        n_repeats=20,        # maybe 3 for speed; you can increase
                # More flexible regularization:
        l1_ratio= 0.1,#0.5,             # ← BALANCED Lasso/Ridge (was 0.9): 50% Lasso + 50% Ridge keeps more features active
        #0.1,         # Mostly Ridge (keeps features in, just shrinks them)
        alpha_min_ratio=0.01,# Allow alpha to get very small (100x smaller than before)
        #0.1,      # ← LESS aggressive alpha range (was 0.01):  Focuses on less regularized models
        n_alphas=50,
        max_iter=100000,
        n_jobs=-1
    )
)


# End timer
end_time = time.time() # Print elapsed time in seconds
elapsed = end_time - start_time
print(f"Process completed in {elapsed/60:.2f} minutes")

#index	feature	mean_drop_cindex	sd_drop_cindex	n_evals
# Top 20 predictors for death:
styled_table = df_imp_death.head(20).style \
    .background_gradient(subset=['mean_drop_cindex'], cmap='Blues') \
    .format({'mean_drop_cindex': "{:.4f}", 'sd_drop_cindex': "{:.4f}"}) \
    .set_properties(**{'text-align': 'left', 'font-family': 'Arial'})

display(styled_table)

#~3 hrs. #6 mni, 3 repeticiones


=== 5 imputations – 10-fold CV permutation importance ===

=== Baseline CV Uno C-index over imputations & folds ===
Mean ± SD: 0.7106 ± 0.0174
Process completed in 18.45 minutes


,feature,mean_drop_cindex,sd_drop_cindex,n_evals
0,adm_age_rec3,0.1930,0.0268,1000
1,primary_sub_mod_alcohol,0.0036,0.0015,1000
2,eva_ocupacion,0.0035,0.0022,1000
3,prim_sub_freq_rec,0.0019,0.0014,1000
4,eva_fisica,0.0015,0.0017,1000
5,dit_m,0.0012,0.0046,1000
6,tenure_status_household,0.0010,0.0018,1000
7,cohabitation_with_couple_children,0.0006,0.0005,1000
8,eva_sm,0.0003,0.0006,1000
9,any_phys_dx,0.0001,0.0001,1000


In [ ]:
# Start timer
start_time = time.time()

df_importance_death = permutation_importance_fixed_alpha(
    X_list=imputations_list_jan26,  # Your Ordinal Encoded Data
    y_surv_list=y_target_death,         # Corrected Outcome List
    alpha_idx=49,                 # The index that worked in CV
    n_splits=5,
    n_repeats=20
)

# End timer
end_time = time.time() # Print elapsed time in seconds
elapsed = end_time - start_time
print(f"Process completed in {elapsed/60:.2f} minutes")

print("\nTop 15 Predictors for Death:")
styled_table = df_importance_death.head(15).style \
    .background_gradient(subset=['mean_drop_cindex'], cmap='Blues') \
    .format({'mean_drop_cindex': "{:.4f}", 'sd_drop_cindex': "{:.4f}"}) \
    .set_properties(**{'text-align': 'left', 'font-family': 'Arial'})

display(styled_table)
#12 min, 5 repeat; 3 minutes with 1 repeat

Running Permutation Importance (Fixed Alpha 49) on 5 imputations...

=== Results ===
Baseline C-index: 0.7458 (Target: ~0.608)
Process completed in 16.21 minutes

Top 15 Predictors for Death:


,feature,mean_drop_cindex,sd_drop_cindex
0,adm_age_rec3,0.0846,0.0118
1,primary_sub_mod_alcohol,0.0327,0.0070
2,any_phys_dx,0.0095,0.0045
3,tr_outcome_adm_discharge_adm_reasons,0.0069,0.0023
4,prim_sub_freq_rec,0.0045,0.0035
5,occupation_condition_corr24_unemployed,0.0042,0.0023
6,occupation_condition_corr24_inactive,0.0038,0.0032
7,eva_ocupacion,0.0034,0.0024
8,sex_rec_woman,0.0020,0.0023
9,eva_fisica,0.0018,0.0033


# Landmark

In [ ]:
# Evluation time points
times_eval_grid = np.array([
    0.5,   # 2 weeks
    1,     # 1 month
    3,     # 3 months
    6,     # 6 months
    12,    # 1 year
    36,    # 3 years
    60,    # 5 years
    120    # 10 years
])

# Filtrar solo tiempos dentro de tu rango de datos
max_time = np.max([y['time'].max() for y in y_surv_death_list])
times_eval_grid = times_eval_grid[times_eval_grid <= max_time]

In [ ]:
#@title ⏱️ Time-Specific Importance & Performance (Ranking & Calibration) { display-mode: "form" }

import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sksurv.linear_model import CoxnetSurvivalAnalysis
from sksurv.metrics import concordance_index_ipcw, cumulative_dynamic_auc, brier_score
from joblib import Parallel, delayed

def permutation_importance_timespecific_cv_mi(
    X_list,
    y_surv_list,
    times_eval,  # Array of time points to evaluate
    alpha_idx=25,
    n_splits=5,
    n_repeats=3,
    random_state=2125,
    l1_ratio=0.1,
    alpha_min_ratio=0.001,
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1,
):
    """
    Multiple-imputation + k-fold CV permutation importance for Coxnet
    with time-specific performance metrics (Corrected for Brier Score).
    """
    # Convert to NumPy arrays upfront for speed
    feature_names = X_list[0].columns.tolist()
    n_features = len(feature_names)
    X_list = [X.values.astype(float) for X in X_list]
    n_imputations = len(X_list)
    n = X_list[0].shape[0]

    # Filter times_eval to be within data range
    max_time = np.max([y['time'].max() for y in y_surv_list])
    times_eval = np.array(times_eval)
    times_eval = times_eval[times_eval <= max_time]
    n_times = len(times_eval)

    # Precompute CV splits once
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    cv_splits = list(kf.split(np.arange(n)))

    def format_time_label(months):
        """Convert months to readable label."""
        if months < 1:
            return f"{int(months*4)}wk"
        elif months < 12:
            return f"{int(months)}mo"
        else:
            return f"{int(months/12)}yr"

    # Function to compute all metrics for one imputation-fold
    def compute_fold(d, fold_idx, train_idx, test_idx):
        # print(f"  Imputation {d+1}/{n_imputations}, fold {fold_idx+1}/{n_splits}")
        X_imp = X_list[d]
        X_train = X_imp[train_idx, :]
        X_test = X_imp[test_idx, :]
        y_train = y_surv_list[d][train_idx]
        y_test = y_surv_list[d][test_idx]

        # Local RNG with unique seed
        local_rng = np.random.RandomState(random_state + d * n_splits + fold_idx)

        # Fit Coxnet
        model = CoxnetSurvivalAnalysis(
            l1_ratio=l1_ratio,
            alpha_min_ratio=alpha_min_ratio,
            n_alphas=n_alphas,
            normalize=False,
            fit_baseline_model=True,  # REQUIRED for predict_survival_function
            max_iter=max_iter,
            verbose=False,
        )
        model.fit(X_train, y_train)

        # Effective alpha index
        eff_alpha_idx = min(alpha_idx, len(model.alphas_) - 1)

        # --- PREDICTION STEP (FIXED) ---

        # 1. RISK SCORES (Linear Predictor): Unbounded (-inf to +inf).
        # Use this for Ranking metrics: C-Index, AUC
        risk_scores = model.predict(X_test, alpha=model.alphas_[eff_alpha_idx])

        # 2. SURVIVAL PROBABILITIES: Bounded (0 to 1).
        # Use this for Calibration metrics: Brier Score
        # This returns an array of StepFunctions
        surv_funcs = model.predict_survival_function(X_test, alpha=model.alphas_[eff_alpha_idx])

        # Evaluate the step functions at the specific time points to get probabilities
        # Shape becomes (n_samples, n_times)
        surv_probs = np.row_stack([fn(times_eval) for fn in surv_funcs])

        # --- METRICS CALCULATION ---

        # 1. Global C-index (Uses Risk Scores)
        res_base = concordance_index_ipcw(y_train, y_test, risk_scores)
        cindex_baseline = float(res_base[0])

        # 2. Time-specific AUC (Uses Risk Scores)
        try:
            auc_scores, mean_auc = cumulative_dynamic_auc(
                y_train, y_test, risk_scores, times=times_eval
            )
        except Exception as e:
            # print(f"    Warning: AUC calculation failed - {str(e)}")
            auc_scores = np.full(n_times, np.nan)

        # 3. Time-specific Brier Score (Uses Survival Probabilities)
        # Note: We pass 'surv_probs' here, not 'risk_scores'
        try:
            _, bs_scores = brier_score(y_train, y_test, surv_probs, times=times_eval)
        except Exception as e:
            # print(f"    Warning: BS calculation failed - {str(e)}")
            bs_scores = np.full(n_times, np.nan)

        # 4. Permutation importance (C-index based)
        fold_drops = [[] for _ in range(n_features)]
        for col_idx in range(n_features):
            for r in range(n_repeats):
                X_perm = X_test.copy()
                X_perm[:, col_idx] = local_rng.permutation(X_perm[:, col_idx])

                # Use Risk Scores for permutation ranking check
                risk_perm = model.predict(X_perm, alpha=model.alphas_[eff_alpha_idx])
                res_perm = concordance_index_ipcw(y_train, y_test, risk_perm)
                cindex_perm = float(res_perm[0])
                fold_drops[col_idx].append(cindex_baseline - cindex_perm)

        return {
            'cindex': cindex_baseline,
            'auc_scores': auc_scores,
            'bs_scores': bs_scores,
            'fold_drops': fold_drops
        }

    print(f"\n{'='*80}")
    print(f"TIME-SPECIFIC PERFORMANCE EVALUATION")
    print(f"{'='*80}")
    print(f"Imputations: {n_imputations}, CV folds: {n_splits}")
    print(f"Time points: {len(times_eval)}")
    print(f"Times (months): {times_eval}")
    print(f"{'='*80}\n")

    # Parallelize over all imputation-fold combinations
    results = Parallel(n_jobs=n_jobs)(
        delayed(compute_fold)(d, fold_idx, train_idx, test_idx)
        for d in range(n_imputations)
        for fold_idx, (train_idx, test_idx) in enumerate(cv_splits)
    )

    # Collect global C-index
    baseline_cindices = [res['cindex'] for res in results]

    # Collect time-specific metrics
    all_auc_scores = np.array([res['auc_scores'] for res in results])  # shape: (n_folds, n_times)
    all_bs_scores = np.array([res['bs_scores'] for res in results])

    # Collect permutation importance
    global_drops = [[] for _ in range(n_features)]
    for res in results:
        fold_drops = res['fold_drops']
        for col_idx in range(n_features):
            global_drops[col_idx].extend(fold_drops[col_idx])

    # ==================== AGGREGATE GLOBAL METRICS ====================
    cindex_mean = np.mean(baseline_cindices)
    cindex_sd = np.std(baseline_cindices, ddof=1) if len(baseline_cindices) > 1 else 0.0

    # Mean AUC across all times and folds
    mean_auc_global = np.nanmean(all_auc_scores)
    sd_auc_global = np.nanstd(all_auc_scores, ddof=1)

    # Mean BS across all times and folds
    mean_bs_global = np.nanmean(all_bs_scores)
    sd_bs_global = np.nanstd(all_bs_scores, ddof=1)

    df_global = pd.DataFrame([{
        'cindex_mean': cindex_mean,
        'cindex_sd': cindex_sd,
        'auc_mean_global': mean_auc_global,
        'auc_sd_global': sd_auc_global,
        'bs_mean_global': mean_bs_global,
        'bs_sd_global': sd_bs_global,
        'n_folds_total': len(baseline_cindices),
        'n_timepoints': n_times
    }])

    # ==================== AGGREGATE TIME-SPECIFIC METRICS ====================
    time_rows = []
    for i, t in enumerate(times_eval):
        auc_at_t = all_auc_scores[:, i]
        bs_at_t = all_bs_scores[:, i]

        time_rows.append({
            'time_months': t,
            'time_label': format_time_label(t),
            'auc_mean': np.nanmean(auc_at_t),
            'auc_sd': np.nanstd(auc_at_t, ddof=1),
            'auc_min': np.nanmin(auc_at_t),
            'auc_max': np.nanmax(auc_at_t),
            'bs_mean': np.nanmean(bs_at_t),
            'bs_sd': np.nanstd(bs_at_t, ddof=1),
            'bs_min': np.nanmin(bs_at_t),
            'bs_max': np.nanmax(bs_at_t),
            'n_evals': np.sum(~np.isnan(auc_at_t))
        })

    df_time = pd.DataFrame(time_rows)

    # ==================== AGGREGATE FEATURE IMPORTANCE ====================
    imp_rows = []
    for col_idx in range(n_features):
        arr = np.array(global_drops[col_idx])
        mean_drop = float(arr.mean()) if arr.size > 0 else np.nan
        sd_drop = float(arr.std(ddof=1)) if arr.size > 1 else 0.0
        imp_rows.append({
            "feature": feature_names[col_idx],
            "mean_drop_cindex": mean_drop,
            "sd_drop_cindex": sd_drop,
            "n_evals": int(arr.size),
        })

    df_imp = pd.DataFrame(imp_rows)
    df_imp = df_imp.sort_values("mean_drop_cindex", ascending=False).reset_index(drop=True)

    # ==================== PRINT SUMMARY ====================
    print(f"\n{'='*80}")
    print(f"RESULTS SUMMARY")
    print(f"{'='*80}")
    print(f"\n>>> GLOBAL METRICS (averaged across all times) <<<")
    print(f"C-index:       {cindex_mean:.4f} ± {cindex_sd:.4f}")
    print(f"Mean AUC(t):   {mean_auc_global:.4f} ± {sd_auc_global:.4f}")
    print(f"Mean BS(t):    {mean_bs_global:.4f} ± {sd_bs_global:.4f}")

    print(f"\n>>> TIME-SPECIFIC METRICS <<<")
    print(df_time[['time_label', 'auc_mean', 'auc_sd', 'bs_mean', 'bs_sd']].to_string(index=False))

    print(f"\n>>> TOP 10 MOST IMPORTANT FEATURES (by C-index drop) <<<")
    print(df_imp.head(10).to_string(index=False))
    print(f"{'='*80}\n")

    return df_global, df_time, df_imp

apply

In [ ]:
# Start timer
start_time = time.time()

# Run analysis
df_global, df_time, df_importance = permutation_importance_timespecific_cv_mi(
    X_list=imputations_list_jan26,
    y_surv_list=y_surv_readm_list_corrected,
    times_eval=times_eval_grid,
    alpha_idx=49,  # Use last alpha (least regularization)
    n_splits=5,
    n_repeats=20,
    random_state=2125,
    # Corrected parameters to avoid model collapse
    l1_ratio=0.1,              # More Ridge
    alpha_min_ratio=0.01,     # Allow smaller alphas
    n_alphas=50,
    max_iter=100000,
    n_jobs=-1
)


# End timer
end_time = time.time() # Print elapsed time in seconds
elapsed = end_time - start_time
print(f"Process completed in {elapsed/60:.2f} minutes")

# Display results
print("\n" + "="*80)
print("DETAILED GLOBAL METRICS")
print("="*80)
print(df_global.T)

print("\n" + "="*80)
print("DETAILED TIME-SPECIFIC PERFORMANCE")
print("="*80)
print(df_time)

print("\n" + "="*80)
print("TOP 20 FEATURES")
print("="*80)

styled_table = df_importance.head(20).style \
    .background_gradient(subset=['mean_drop_cindex'], cmap='Blues') \
    .format({'mean_drop_cindex ': "{:.4f}", 'sd_drop_cindex': "{:.4f}"}) \
    .set_properties(**{'text-align': 'left', 'font-family': 'Arial'})

display(styled_table)



#17


TIME-SPECIFIC PERFORMANCE EVALUATION
Imputations: 5, CV folds: 5
Time points: 8
Times (months): [  0.5   1.    3.    6.   12.   36.   60.  120. ]


RESULTS SUMMARY

>>> GLOBAL METRICS (averaged across all times) <<<
C-index:       0.5852 ± 0.0029
Mean AUC(t):   0.6888 ± 0.1001
Mean BS(t):    0.0780 ± 0.0815

>>> TIME-SPECIFIC METRICS <<<
time_label  auc_mean   auc_sd  bs_mean    bs_sd
       2wk  0.764389 0.073513 0.000441 0.000156
       1mo  0.796113 0.049126 0.001378 0.000369
       3mo  0.776517 0.027678 0.007091 0.000321
       6mo  0.751451 0.006827 0.022060 0.001076
       1yr  0.696899 0.006419 0.059455 0.001705
       3yr  0.613649 0.007426 0.143035 0.002852
       5yr  0.580468 0.004304 0.177174 0.001576
      10yr  0.531187 0.013521 0.213420 0.002054

>>> TOP 10 MOST IMPORTANT FEATURES (by C-index drop) <<<
                             feature  mean_drop_cindex  sd_drop_cindex  n_evals
                        adm_age_rec3          0.012224        0.003547      500
         

,feature,mean_drop_cindex,sd_drop_cindex,n_evals
0,adm_age_rec3,0.012224,0.0035,500
1,sex_rec_woman,0.008964,0.0020,500
2,plan_type_corr_pg_pr,0.004675,0.0014,500
3,dit_m,0.004215,0.0030,500
4,plan_type_corr_m_pr,0.002925,0.0008,500
5,ethnicity,0.002817,0.0006,500
6,dg_psiq_cie_10_dg,0.002300,0.0008,500
7,primary_sub_mod_alcohol,0.002274,0.0013,500
8,sub_dep_icd10_status_drug_dependence,0.001756,0.0007,500
9,prim_sub_freq_rec,0.001367,0.0010,500


In [ ]:
#@title 📥 Download Analysis Results (Excel) { display-mode: "form" }
#@markdown Run this cell after the analysis is finished to download the data.

file_name_prefix = "Coxnet_Readmission_Study_cindex" #@param {type:"string"}
add_timestamp = True #@param {type:"boolean"}

from google.colab import files
import pandas as pd
from datetime import datetime

def trigger_download(df_glob, df_t, df_imp):
    # 1. Handle naming
    timestamp = datetime.now().strftime("_%Y%m%d_%H%M") if add_timestamp else ""
    full_filename = f"{file_name_prefix}{timestamp}.xlsx"

    # 2. Ensure engine is installed
    try:
        import xlsxwriter
    except ImportError:
        !pip install -q xlsxwriter

    # 3. Create Excel with multiple sheets
    with pd.ExcelWriter(full_filename, engine='xlsxwriter') as writer:
        df_glob.to_excel(writer, sheet_name='Global_Metrics', index=False)
        df_t.to_excel(writer, sheet_name='Performance_Over_Time', index=False)
        df_imp.to_excel(writer, sheet_name='Permutation_Importance', index=False)

        # Aesthetic touch: Auto-adjust column widths
        workbook = writer.book
        for sheetname in writer.sheets:
            worksheet = writer.sheets[sheetname]
            worksheet.set_column('A:Z', 22) # Set width to 22 for all columns

    print(f"✅ Excel file '{full_filename}' generated successfully.")
    files.download(full_filename)

# Verification check
try:
    # We use the specific variables returned by your function
    trigger_download(df_global, df_time, df_importance)
except NameError:
    print("❌ Error: Results not found in memory. Please run the analysis cell first.")
except Exception as e:
    print(f"❌ An error occurred: {e}")

✅ Excel file 'Coxnet_Readmission_Study_cindex_20260122_1553.xlsx' generated successfully.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Run analysis
df_global_death, df_time_death, df_importance_death = permutation_importance_timespecific_cv_mi(
  X_list=imputations_list_jan26,
  y_surv_list=y_surv_death_list,
  times_eval=times_eval_grid,
  alpha_idx=49,  # Use last alpha (least regularization)
  n_splits=10,
  n_repeats=20,
  random_state=2125,
  # Corrected parameters to avoid model collapse
  l1_ratio=0.1,              # More Ridge
  alpha_min_ratio=0.01,     # Allow smaller alphas
  n_alphas=50,
  max_iter=100000,
  n_jobs=-1
)

# Display results
print("\n" + "="*80)
print("DETAILED GLOBAL METRICS")
print("="*80)
print(df_global_death.T)

print("\n" + "="*80)
print("DETAILED TIME-SPECIFIC PERFORMANCE")
print("="*80)
print(df_time_death)

print("\n" + "="*80)
print("TOP 20 FEATURES")
print("="*80)

styled_table = df_importance_death.head(20).style \
    .background_gradient(subset=['mean_drop_cindex'], cmap='Blues') \
    .format({'mean_drop_cindex': "{:.4f}", 'sd_drop_cindex': "{:.4f}"}) \
    .set_properties(**{'text-align': 'left', 'font-family': 'Arial'})

display(styled_table)


#17 min


TIME-SPECIFIC PERFORMANCE EVALUATION
Imputations: 5, CV folds: 10
Time points: 8
Times (months): [  0.5   1.    3.    6.   12.   36.   60.  120. ]


RESULTS SUMMARY

>>> GLOBAL METRICS (averaged across all times) <<<
C-index:       0.7106 ± 0.0174
Mean AUC(t):   0.7399 ± 0.0653
Mean BS(t):    0.0178 ± 0.0232

>>> TIME-SPECIFIC METRICS <<<
time_label  auc_mean   auc_sd  bs_mean    bs_sd
       2wk  0.720098 0.159546 0.000124 0.000148
       1mo  0.758296 0.101050 0.000328 0.000129
       3mo  0.747555 0.052906 0.001428 0.000395
       6mo  0.724674 0.048674 0.003483 0.000526
       1yr  0.735512 0.029726 0.007036 0.000897
       3yr  0.735603 0.015290 0.022802 0.000909
       5yr  0.745161 0.012540 0.037846 0.001255
      10yr  0.744511 0.016312 0.069244 0.003704

>>> TOP 10 MOST IMPORTANT FEATURES (by C-index drop) <<<
                          feature  mean_drop_cindex  sd_drop_cindex  n_evals
                     adm_age_rec3          0.192965        0.026831     1000
          prim

,feature,mean_drop_cindex,sd_drop_cindex,n_evals
0,adm_age_rec3,0.1930,0.0268,1000
1,primary_sub_mod_alcohol,0.0036,0.0015,1000
2,eva_ocupacion,0.0035,0.0022,1000
3,prim_sub_freq_rec,0.0019,0.0014,1000
4,eva_fisica,0.0015,0.0017,1000
5,dit_m,0.0012,0.0046,1000
6,tenure_status_household,0.0010,0.0018,1000
7,cohabitation_with_couple_children,0.0006,0.0005,1000
8,eva_sm,0.0003,0.0006,1000
9,any_phys_dx,0.0001,0.0001,1000


In [ ]:
#@title 📥 Export Death Outcome Results { display-mode: "form" }
#@markdown Run this cell once the 17-minute analysis is finished to download the results.

file_prefix = "Coxnet_Mortality_Study_cindex" #@param {type:"string"}
include_timestamp = True #@param {type:"boolean"}

from google.colab import files
import pandas as pd
from datetime import datetime

def export_death_results(df_glob, df_t, df_imp):
    # 1. Filename Setup
    ts = datetime.now().strftime("_%Y%m%d_%H%M") if include_timestamp else ""
    filename = f"{file_prefix}{ts}.xlsx"

    # 2. Ensure dependencies
    try:
        import xlsxwriter
    except ImportError:
        !pip install -q xlsxwriter

    # 3. Create Excel Workbook
    with pd.ExcelWriter(filename, engine='xlsxwriter') as writer:
        df_glob.to_excel(writer, sheet_name='Global_Metrics', index=True)
        df_t.to_excel(writer, sheet_name='Time_Specific_Metrics', index=False)
        df_imp.to_excel(writer, sheet_name='Feature_Importance', index=False)

        # Professional formatting
        workbook = writer.book
        for sheetname in writer.sheets:
            worksheet = writer.sheets[sheetname]
            worksheet.set_column('A:A', 30) # Feature names usually need more space
            worksheet.set_column('B:Z', 18)

    print(f"✅ Excel file '{filename}' created.")
    files.download(filename)

# Execution logic
try:
    export_death_results(df_global_death, df_time_death, df_importance_death)
except NameError:
    print("❌ Error: 'df_global_death' not found. Ensure the analysis finished running.")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ Excel file 'Coxnet_Mortality_Study_cindex_20260122_1610.xlsx' created.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Visualize

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_time_dependent_performance(df, outcome_name="Readmission"):
    """
    Flexible visualization for time-specific AUC and Brier Score.
    Uses the 'df' argument dynamically so it works for both Readmission and Death.
    """
    # Create figure
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # --- Plot 1: Discrimination (AUC) ---
    # We use 'df' here, NOT df_time or df_time_death
    axes[0].errorbar(
        df['time_months'],
        df['auc_mean'],
        yerr=df['auc_sd'],
        marker='o', markersize=8, capsize=6, linewidth=2.5,
        color='steelblue', ecolor='lightblue', label='Mean AUC(t) ± SD'
    )
    axes[0].fill_between(
        df['time_months'],
        df['auc_min'],
        df['auc_max'],
        alpha=0.15, color='steelblue', label='Min-Max Range'
    )

    # Reference lines
    axes[0].axhline(0.5, color='red', linestyle='--', linewidth=2, label='Random Chance')
    axes[0].axhline(0.7, color='green', linestyle=':', linewidth=1.5, label='Good Discrimination')

    # Labels & Titles
    axes[0].set_xlabel('Follow-up Time (months)', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('AUC(t)', fontsize=13, fontweight='bold')
    axes[0].set_title(f'Discrimination (AUC) - {outcome_name}', fontsize=15, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(alpha=0.3, linestyle='--')
    axes[0].set_ylim([0.4, 1.0])

    # X-Axis Ticks
    axes[0].set_xticks(df['time_months'])
    axes[0].set_xticklabels(df['time_label'], rotation=45, ha='right')

    # --- Plot 2: Calibration (Brier Score) ---
    axes[1].errorbar(
        df['time_months'],
        df['bs_mean'],
        yerr=df['bs_sd'],
        marker='s', markersize=8, capsize=6, linewidth=2.5,
        color='darkorange', ecolor='bisque', label='Mean BS(t) ± SD'
    )
    axes[1].fill_between(
        df['time_months'],
        df['bs_min'],
        df['bs_max'],
        alpha=0.15, color='darkorange', label='Min-Max Range'
    )

    # Labels & Titles
    axes[1].set_xlabel('Follow-up Time (months)', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Brier Score(t)', fontsize=13, fontweight='bold')
    axes[1].set_title(f'Calibration (Brier Score) - {outcome_name}', fontsize=15, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(alpha=0.3, linestyle='--')

    # X-Axis Ticks
    axes[1].set_xticks(df['time_months'])
    axes[1].set_xticklabels(df['time_label'], rotation=45, ha='right')

    plt.tight_layout()
    return fig

In [ ]:
#@title 📥 Export Both Plots (Readmission & Death) { display-mode: "form" }
dpi_quality = 300 #@param {type:"integer"}
file_format = "png" #@param ["png", "pdf", "svg"]

from google.colab import files
import matplotlib.pyplot as plt

def bulk_export_plots():
    tasks = []

    # Check GLOBAL memory for the dataframes
    if 'df_time' in globals():
        tasks.append((globals()['df_time'], "Readmission"))
    else:
        print("⚠️ Warning: 'df_time' (Readmission) not found.")

    if 'df_time_death' in globals():
        tasks.append((globals()['df_time_death'], "Mortality"))
    else:
        print("⚠️ Warning: 'df_time_death' (Mortality) not found.")

    if not tasks:
        print("❌ No data found to plot. Please run the analysis cells first.")
        return

    for df_obj, label in tasks:
        print(f"🎨 Generating plot for: {label}...")

        # 1. Generate the plot
        # Ensure your plotting function is called and returns a figure object
        plot_time_dependent_performance(df_obj, outcome_name=label)
        fig = plt.gcf() # Get Current Figure

        # 2. Save
        filename = f"Plot_Performance_{label}_{dpi_quality}dpi.{file_format}"
        fig.savefig(filename, bbox_inches='tight', dpi=dpi_quality)

        # 3. Download
        files.download(filename)
        print(f"✅ Downloaded: {filename}")

        # 4. Clean up
        plt.close(fig)

try:
    bulk_export_plots()
except Exception as e:
    print(f"❌ Error during export: {e}")

🎨 Generating plot for: Readmission...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: Plot_Performance_Readmission_300dpi.png
🎨 Generating plot for: Mortality...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: Plot_Performance_Mortality_300dpi.png
